# 06 · Demo do modelo binário pré-treinado (PT-BR)

- **Origem:** `model/example.ipynb`
- **Faz:** baixa o modelo binário pré-treinado distribuído pelo autor e roda uma inferência de exemplo (`"este é um exemplo."`).
- **Entradas:** download externo do Google Drive (id `1Q8MuO4SsND0xzDIW9TNvzfl5Fc2NGwAJ`).
- **Saídas:** predição de exemplo (classe + logits).

> **Atenção — execução condicionada a download externo.** Se o link do Google Drive (2020) estiver fora do ar, considere este notebook **NÃO-EXECUTÁVEL** — isso não afeta os demais notebooks, que treinam o modelo do zero.

> **Diff mínimo.** Cada alteração abaixo é marcada **[T]** tecnologia descontinuada, **[B]** bug ou **[A]** fidelidade ao artigo — catálogo completo em [`CHANGES.md`](CHANGES.md).

In [1]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'ToLD-BR.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_ZIP  = ROOT / 'experiments' / 'data' / '1annotator.zip'
MAIN_CSV  = ROOT / 'ToLD-BR.csv'
ALPHA_CSV = ROOT / 'ToLD-BR_alpha.csv'
RESULTS   = ROOT / 'reproduction' / 'results'
FIGURES   = RESULTS / 'figures'
RESULTS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

**Mudança #1 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | remoção da instalação de dependências via shell |
| **Antes** | célula `%%writefile setup.sh` (clone de NVIDIA/apex, `pip install simpletransformers/unidecode/gdown`) seguida de `!sh setup.sh` |
| **Depois** | célula de `import` (`zipfile`, `torch`, `gdown`, `transformers`) — instalação fora do notebook |
| **Motivo** | apex (mixed-precision manual) e simpletransformers são tecnologia descontinuada/desnecessária — o fp16 nativo vem do PyTorch/`transformers`, e a inferência usa `transformers` puro. Instalação de pacotes não pertence ao corpo do notebook |
| **Impacto** | nenhuma mudança de resultado; dependências (`transformers`, `torch`, `gdown`) são pré-requisitos do ambiente da reprodução |

In [2]:
import zipfile
import torch
import gdown
from transformers import AutoTokenizer, AutoModelForSequenceClassification

c:\Users\Pedro\AppData\Local\toldbr-repro-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Mudança #2 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | download do modelo pré-treinado sem `google.colab` |
| **Antes** | `from google.colab import drive, files` + `gdown.download(url, 'toxic_bert_model.zip', quiet=True)` + `os.environ['modelpath'] = './toxic_bert_model.zip'` |
| **Depois** | `gdown.download(url, str(model_zip), ...)` para `RESULTS/toxic_bert_model.zip` (mesmo id), sem `google.colab` nem env var |
| **Motivo** | `google.colab` não existe fora do Colab; `os.environ['modelpath']` só servia ao comando shell. Mantém-se o `gdown` (funciona no Windows) com o mesmo id `1Q8MuO4SsND0xzDIW9TNvzfl5Fc2NGwAJ`. O binário do modelo (~400 MB) não é versionado no repositório, daí o download externo |
| **Impacto** | nenhum — mesmo arquivo `toxic_bert_model.zip` baixado; se o link do Drive cair, este notebook fica não-executável (ver aviso no topo) |

In [3]:
# Download externo do modelo pré-treinado (binário não versionado no repositório).
# Se este link cair, marque o notebook como não-executável (ver aviso no topo).
url = 'https://drive.google.com/uc?id=1Q8MuO4SsND0xzDIW9TNvzfl5Fc2NGwAJ'
model_zip = RESULTS / 'toxic_bert_model.zip'
model_dir = RESULTS / 'toxic_bert_model'

if not model_dir.exists():
    gdown.download(url, str(model_zip), quiet=False)
    print('Download concluído:', model_zip)
else:
    print('Modelo já presente em:', model_dir)

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1Q8MuO4SsND0xzDIW9TNvzfl5Fc2NGwAJ

but Gdown can't. Please check connections and permissions.

**Mudança #3 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | descompactação do modelo com Python `zipfile` em vez de `!unzip` |
| **Antes** | `!unzip "$modelpath" -d .` |
| **Depois** | `zipfile.ZipFile(model_zip).extractall(RESULTS)` |
| **Motivo** | `unzip` é um binário Unix que não existe por padrão no Windows; `zipfile` é multiplataforma e parte da stdlib |
| **Impacto** | nenhum — mesma estrutura `toxic_bert_model/` extraída (vocab.txt, config.json, pytorch_model.bin, etc.) |

In [4]:
# Extração multiplataforma do modelo (substitui o !unzip Unix do original).
if not (model_dir / 'config.json').exists():
    with zipfile.ZipFile(model_zip, 'r') as zf:
        zf.extractall(RESULTS)
    print('Modelo extraído em:', model_dir)
else:
    print('Modelo já extraído em:', model_dir)

# Conferência: lista os arquivos do modelo extraído.
for f in sorted(p.name for p in model_dir.iterdir()):
    print(' -', f)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Pedro\\OneDrive\\Documentos\\UFCG\\Mestrado\\2026.1\\Projeto - Reprodução\\ToLD-Br\\reproduction\\results\\toxic_bert_model.zip'

**Mudança #4 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | carregamento e inferência via `transformers` + `torch` puro |
| **Antes** | `ClassificationModel('distilbert', 'toxic_bert_model')` + `model.predict(['este é um exemplo.'])` (simpletransformers) |
| **Depois** | `AutoTokenizer`/`AutoModelForSequenceClassification` + inferência `torch` (logits → `argmax`); arquitetura lida do `config.json` |
| **Motivo** | simpletransformers é a tecnologia descontinuada que esta reprodução substitui por HuggingFace `transformers`. O modelo distribuído já está em formato HuggingFace (`config.json` + `pytorch_model.bin`), então é carregado direto; a arquitetura é lida do próprio `config.json` (não precisa especificar `'distilbert'`) |
| **Impacto** | MESMA predição (`0` = não-tóxico) e mesmos logits do original (`[[1.8342592, -1.7641641]]`), dentro da tolerância de ponto flutuante |

In [ ]:
# Carrega tokenizer e modelo no formato HuggingFace (a arquitetura vem do config.json).
tokenizer = AutoTokenizer.from_pretrained(str(model_dir), do_lower_case=False)
model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
model.eval()

# Inferência reproduzindo simpletransformers.ClassificationModel.predict():
#   - predictions = argmax dos logits
#   - outputs     = logits brutos
texts = ['este é um exemplo.']
enc = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
with torch.no_grad():
    logits = model(**enc).logits

predictions = logits.argmax(dim=-1).tolist()
outputs = logits.tolist()

print('Predictions:', predictions)
print('Outputs:', outputs)
print('0 = não-tóxico' if predictions[0] == 0 else '1 = tóxico')